# PINN derivatives with PyTorch autograd

This notebook shows how to compute

$$
u_t,\quad u_x,\quad u_{xx}
$$

using PyTorch automatic differentiation.

We also extend the idea to 2D and 3D spatial inputs.

In [1]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


### 1. One-dimensional case: $ u_\theta(x,t) $

We first define a neural network

$$
u_\theta(x,t)\approx u(x,t).
$$

The model input is \((x,t)\), and the output is \(u\).

In [24]:
class PINN1D(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        return self.net(xt)


model_1d = PINN1D().to(device)

## 2. Sample points

To compute derivatives with respect to \(x\) and \(t\), we set

```python
x.requires_grad_(True)
t.requires_grad_(True)
```

Here we evaluate the model at

$$
x\in[-1,1], \qquad t=0.5.
$$

In [25]:
N = 100

x = torch.linspace(-1.0, 1.0, N).view(-1, 1).to(device)
t = torch.ones_like(x).to(device) * 0.5

x.requires_grad_(True)
t.requires_grad_(True)

u = model_1d(x, t)

# print("x shape:", x.shape)
# print("t shape:", t.shape)
# print("u shape:", u.shape)

#x

### 3. Compute $ u_t$ and $u_x$

The output \(u\) is not a scalar. It has shape \((N,1)\).  
Therefore, we use

```python
grad_outputs=torch.ones_like(u)
```

This means PyTorch differentiates

$$
\sum_{i=1}^N u_i.
$$

For a usual MLP PINN, each output \(u_i\) depends only on its own input \((x_i,t_i)\).  
Therefore, this gives the pointwise derivatives.

In [26]:
# x.shape, t.shape, u.shape
#x[0]

xt = torch.cat([x, t], dim=1)
x.shape, t.shape, xt.shape

#xt

#t

(torch.Size([100, 1]), torch.Size([100, 1]), torch.Size([100, 2]))

In [28]:
u_t = torch.autograd.grad(
    u,
    t,
    grad_outputs=torch.ones_like(u),
    create_graph=True
)[0]

#print("u_t shape:", u_t.shape)

#u_t
u_x = torch.autograd.grad(
    u,
    x,
    grad_outputs=torch.ones_like(u),
    create_graph=True
)[0]


#u_x
# print("u_x shape:", u_x.shape)

## 4. Compute $u_{xx} $

Now differentiate \(u_x\) again with respect to \(x\):

$$
u_{xx}
=
\frac{\partial}{\partial x}
\left(
\frac{\partial u}{\partial x}
\right).
$$

In [ ]:
u_xx = torch.autograd.grad(
    u_x,
    x,
    grad_outputs=torch.ones_like(u_x),
    create_graph=True
)[0]

print("u_xx shape:", u_xx.shape)


u_xx shape: torch.Size([100, 1])


tensor([[ 0.0789],
        [ 0.0788],
        [ 0.0786],
        [ 0.0784],
        [ 0.0782],
        [ 0.0778],
        [ 0.0774],
        [ 0.0770],
        [ 0.0765],
        [ 0.0759],
        [ 0.0752],
        [ 0.0745],
        [ 0.0737],
        [ 0.0728],
        [ 0.0719],
        [ 0.0709],
        [ 0.0698],
        [ 0.0686],
        [ 0.0674],
        [ 0.0661],
        [ 0.0647],
        [ 0.0632],
        [ 0.0616],
        [ 0.0600],
        [ 0.0583],
        [ 0.0565],
        [ 0.0546],
        [ 0.0527],
        [ 0.0506],
        [ 0.0485],
        [ 0.0463],
        [ 0.0441],
        [ 0.0418],
        [ 0.0394],
        [ 0.0369],
        [ 0.0343],
        [ 0.0317],
        [ 0.0291],
        [ 0.0263],
        [ 0.0236],
        [ 0.0207],
        [ 0.0178],
        [ 0.0149],
        [ 0.0119],
        [ 0.0089],
        [ 0.0058],
        [ 0.0027],
        [-0.0005],
        [-0.0036],
        [-0.0068],
        [-0.0100],
        [-0.0133],
        [-0.

## 5. Burgers equation residual

The viscous Burgers equation is

$$
u_t + u u_x = \nu u_{xx}.
$$

Therefore the PINN residual is

$$
r_\theta(x,t)
=
u_t + u u_x - \nu u_{xx}.
$$

In [ ]:
nu = 0.01 / np.pi

residual = u_t + u * u_x - nu * u_xx

print("residual shape:", residual.shape)
print("mean residual squared:", torch.mean(residual**2).item())

## 6. Mathematical meaning of `grad_outputs`

The code

```python
torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u))
```

computes

$$
\frac{\partial}{\partial x}
\sum_{i=1}^N u_i.
$$

The \(j\)-th component is

$$
\frac{\partial}{\partial x_j}
\sum_{i=1}^N u_i
=
\sum_{i=1}^N
\frac{\partial u_i}{\partial x_j}.
$$

For a standard MLP PINN,

$$
u_i = u_\theta(x_i,t_i).
$$

Thus,

$$
\frac{\partial u_i}{\partial x_j}=0
\quad \text{if } i\neq j.
$$

So

$$
\frac{\partial}{\partial x_j}
\sum_{i=1}^N u_i
=
\frac{\partial u_j}{\partial x_j}.
$$

The same logic works for the second derivative:

$$
\frac{\partial}{\partial x_j}
\sum_{i=1}^N
\frac{\partial u_i}{\partial x_i}
=
\frac{\partial^2 u_j}{\partial x_j^2}.
$$

## 2D spatial input: $ u_\theta(x,y,t) $

Now consider

$$
u_\theta(x,y,t).
$$

We compute

$$
u_t,\quad u_x,\quad u_y,\quad u_{xx},\quad u_{yy}.
$$

In [7]:
class PINN2D(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, y, t):
        xyt = torch.cat([x, y, t], dim=1)
        return self.net(xyt)


model_2d = PINN2D().to(device)

In [8]:
N = 100

x = torch.rand(N, 1).to(device) * 2.0 - 1.0
y = torch.rand(N, 1).to(device) * 2.0 - 1.0
t = torch.rand(N, 1).to(device)

x.requires_grad_(True)
y.requires_grad_(True)
t.requires_grad_(True)

u = model_2d(x, y, t)

u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
u_y = torch.autograd.grad(u, y, torch.ones_like(u), create_graph=True)[0]
u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]

u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
u_yy = torch.autograd.grad(u_y, y, torch.ones_like(u_y), create_graph=True)[0]

lap_u = u_xx + u_yy

print("u shape:    ", u.shape)
print("u_x shape:  ", u_x.shape)
print("u_y shape:  ", u_y.shape)
print("u_t shape:  ", u_t.shape)
print("u_xx shape: ", u_xx.shape)
print("u_yy shape: ", u_yy.shape)
print("lap_u shape:", lap_u.shape)

u shape:     torch.Size([100, 1])
u_x shape:   torch.Size([100, 1])
u_y shape:   torch.Size([100, 1])
u_t shape:   torch.Size([100, 1])
u_xx shape:  torch.Size([100, 1])
u_yy shape:  torch.Size([100, 1])
lap_u shape: torch.Size([100, 1])


## 2D heat equation residual

For the 2D heat equation,

$$
u_t = \nu (u_{xx}+u_{yy}),
$$

the residual is

$$
r_\theta(x,y,t)
=
u_t - \nu(u_{xx}+u_{yy}).
$$

In [ ]:
nu = 0.01

residual_2d_heat = u_t - nu * (u_xx + u_yy)

print("2D heat residual shape:", residual_2d_heat.shape)
print("mean residual squared:", torch.mean(residual_2d_heat**2).item())

## 3D spatial input: $ u_\theta(x,y,z,t) $

Now consider

$$
u_\theta(x,y,z,t).
$$

The 3D Laplacian is

$$
\Delta u = u_{xx}+u_{yy}+u_{zz}.
$$

In [10]:
class PINN3D(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, y, z, t):
        xyzt = torch.cat([x, y, z, t], dim=1)
        return self.net(xyzt)


model_3d = PINN3D().to(device)

In [11]:
N = 100

x = torch.rand(N, 1).to(device) * 2.0 - 1.0
y = torch.rand(N, 1).to(device) * 2.0 - 1.0
z = torch.rand(N, 1).to(device) * 2.0 - 1.0
t = torch.rand(N, 1).to(device)

x.requires_grad_(True)
y.requires_grad_(True)
z.requires_grad_(True)
t.requires_grad_(True)

u = model_3d(x, y, z, t)

u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
u_y = torch.autograd.grad(u, y, torch.ones_like(u), create_graph=True)[0]
u_z = torch.autograd.grad(u, z, torch.ones_like(u), create_graph=True)[0]
u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]

u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
u_yy = torch.autograd.grad(u_y, y, torch.ones_like(u_y), create_graph=True)[0]
u_zz = torch.autograd.grad(u_z, z, torch.ones_like(u_z), create_graph=True)[0]

lap_u = u_xx + u_yy + u_zz

print("u shape:    ", u.shape)
print("u_x shape:  ", u_x.shape)
print("u_y shape:  ", u_y.shape)
print("u_z shape:  ", u_z.shape)
print("u_t shape:  ", u_t.shape)
print("u_xx shape: ", u_xx.shape)
print("u_yy shape: ", u_yy.shape)
print("u_zz shape: ", u_zz.shape)
print("lap_u shape:", lap_u.shape)

u shape:     torch.Size([100, 1])
u_x shape:   torch.Size([100, 1])
u_y shape:   torch.Size([100, 1])
u_z shape:   torch.Size([100, 1])
u_t shape:   torch.Size([100, 1])
u_xx shape:  torch.Size([100, 1])
u_yy shape:  torch.Size([100, 1])
u_zz shape:  torch.Size([100, 1])
lap_u shape: torch.Size([100, 1])


## Important caution

This method is correct when each output sample depends only on its own input sample:

$$
u_i = u_\theta(x_i,t_i)
$$

or

$$
u_i = u_\theta(x_i,y_i,t_i).
$$

This is true for a standard MLP PINN.

Be careful if the model mixes samples in a batch, for example with:

- batch normalization,
- attention across batch samples,
- graph/message-passing layers across points.

In those cases, the output \(u_i\) may depend on other input samples.

## Summary

For 1D Burgers equation:

$$
u_t + uu_x - \nu u_{xx}=0.
$$

Use:

```python
u_t  = grad(u, t)
u_x  = grad(u, x)
u_xx = grad(u_x, x)
```

For 2D heat equation:

$$
u_t - \nu(u_{xx}+u_{yy})=0.
$$

Use:

```python
lap_u = u_xx + u_yy
```

For 3D heat equation:

$$
u_t - \nu(u_{xx}+u_{yy}+u_{zz})=0.
$$

Use:

```python
lap_u = u_xx + u_yy + u_zz
```